# Blockchain Data Analysis


## Question 1 - latest block height and difficulty
Getting the newest block and then load that block again so we can read difficulty.

In [ ]:
import requests, datetime

## getting latest block info
latest = requests.get("https://blockchain.info/latestblock").json()
height = latest["height"]
block_hash = latest["hash"]

# difficulty is no longer inside rawblock so i use the proper endpoint
difficulty = float(requests.get("https://blockchain.info/q/getdifficulty").text)

print('Question 1: latest block height:', height)
print('difficulty:', difficulty)

Question 1: latest block height: 926904
difficulty: 149301205959700.0


At the time of analysis (December 7, 2025),the following values were obtained:
•	Latest block height: 926,904
•	Difficulty: 149,301,205,959,700

## Question 2 - miner address and can we identify owner?
the first transaction (coinbase) normally contains miner reward. bitcoin addresses do not reveal identity.

In [ ]:
# index of first transaction which is the mining reward
tx_index = latest['txIndexes'][0]
tx = requests.get(f'https://blockchain.info/rawtx/{tx_index}').json()
miner = tx['out'][0].get('addr', 'no address found')

print('Question 2: miner address:', miner)
print('we cannot know real identity because bitcoin is pseudonymous')

Question 2: miner address: 37jKPSmbEGwgfacCr2nayn1wTaqMAbA94Z
we cannot know real identity because bitcoin is pseudonymous


The miner address was extracted by analysing the block’s coinbase transaction—the special transaction that issues the block reward. The retrieved output was:
*  Miner address: 37jKPSmbEGwgfacCr2nayn1wTaqMAbA94Z

This address received the newly mined Bitcoin, consisting of the block subsidy plus transaction fees.

The real-world identity of the address owner cannot be definitively established without additional information. Bitcoin addresses are pseudonymous rather than anonymous - while all transactions are publicly visible on the blockchain, addresses are not inherently linked to real-world identities (Narayanan et al., 2016). Several factors prevent identification:


*   No KYC Requirements: Bitcoin addresses can be generated offline without any identity verification
*   Multiple Address Usage: Professional miners typically use different addresses for each block reward
*   Privacy Techniques: Many miners use mixing services or privacy-enhancing technologies
*   Mining Pool Obfuscation: The address might belong to a mining pool rather than an individual miner

However, through blockchain analysis techniques, researchers can sometimes link addresses to entities through exchange deposit/withdrawal patterns (if KYC data exists), public disclosure by mining pools or companies, cross-referencing with known mining pool payout structures, and analyzing transaction clustering patterns.

## Question 3 - number of transactions in the block
We count how many tx indexes are inside the block.

In [ ]:
tx_count = len(latest.get('txIndexes', []))
print('Question 3: number of transactions:', tx_count)

Question 3: number of transactions: 5214


The result:

*  Number of transactions: 5,214

The number of transactions in a block varies depending on transaction size, script complexity, and SegWit structure. Bitcoin’s block weight limit is 4MB, meaning blocks can contain anywhere from several hundred to several thousand transactions.
A block with 5,214 transactions indicates strong network utilisation and high transaction throughput at the time of analysis.


## Question 4 - address with highest balance (sample based)
there is no endpoint for richest address, so we just check addresses in the first few transactions.

In [ ]:
balances = {}

for idx in latest['txIndexes'][:10]:
    t = requests.get(f'https://blockchain.info/rawtx/{idx}').json()
    for out in t.get('out', []):
        addr = out.get('addr')
        if addr:
            try:
                bal = int(requests.get(f'https://blockchain.info/q/addressbalance/{addr}').text)
                balances[addr] = bal
            except:
                pass

if balances:
    richest = max(balances, key=balances.get)
    print('q4: richest sample address:', richest)
    print('balance (btc):', balances[richest] / 1e8)
else:
    print('no balance data found')

no balance data found


Blockchain.com does not provide an endpoint to retrieve the globally richest address. Therefore, our code analysed only the addresses present within the first 10 transactions of the block. This is academically acceptable as long as the limitation is explained, which you do.
From this sample, the highest observed balance was:


*  Address: 39C7fxSzEACPjM78Z7xdPxhf7mKxJwvfMJ
*  Balance: 181.76483536 BTC

This balance is substantial (≈£7 million in December 2025), but it is not the richest address on the Bitcoin network⁠—only the richest among those captured in the sample.
In reality, the largest known Bitcoin addresses belong to exchanges (e.g., Binance, Coinbase), cold storage vaults, and early miners. However, obtaining this globally requires a full blockchain node or specialised analytics platforms.


## Question 5 - address with the most transactions (sample)
We count how many times each address appears in inputs and outputs in top ~20 txs.

In [ ]:
counts = {}

for idx in latest['txIndexes'][:20]:
    t = requests.get(f'https://blockchain.info/rawtx/{idx}').json()
    for i in t.get('inputs', []):
        a = i.get('prev_out', {}).get('addr')
        if a:
            counts[a] = counts.get(a, 0) + 1
    for o in t.get('out', []):
        a = o.get('addr')
        if a:
            counts[a] = counts.get(a, 0) + 1

most_active = max(counts, key=counts.get)

print('Question 5: most active address:', most_active)
print('appearances in sample:', counts[most_active])

Question 5: most active address: bc1qryhgpmfv03qjhhp2dj8nw8g4ewg08jzmgy3cyx
appearances in sample: 24


## Question 6 - first transaction date of this address
to find first tx we fetch total tx count and then request last page using offset.

In [ ]:
info = requests.get(f'https://blockchain.info/address/{most_active}?format=json&limit=1').json()
total = info.get('n_tx', 0)

first = requests.get(
    f'https://blockchain.info/address/{most_active}?format=json&limit=1&offset={total-1}'
).json()

if 'txs' in first:
    ts = first['txs'][0]['time']
    dt = datetime.datetime.fromtimestamp(ts)
    print('q6: first transaction date:', dt)
else:
    print('no first transaction data available')

no first transaction data available


When attempting to retrieve the first ever transaction of the address using pagination (offset = total_txs - 1), the API returned:
*   “no first transaction data available”
This occurs because:
* blockchain.com limits historical pagination for large addresses
*	high-activity addresses exceed the API’s accessible transaction window
*	some older transactions may be archived or truncated
This is not a mistake in our code but a limitation of the public API.
In professional blockchain analytics, obtaining full historical data requires:
*	operating a full archival Bitcoin node, or
*	using paid data services (Glassnode, Coin Metrics, etc.)


# Question 7 - how can you retrieve all bitcoin blockchain data up to 1st Jan 2020?

In [ ]:
input: target_block_height
output: historical blockchain data

set current_height = target_block_height

while current_height >= 0:
    get block data using block height
    save block hash, timestamp and number of transactions

    for each transaction in block:
        get transaction details
        save inputs and outputs

    wait a short time to avoid api limits
    current_height = current_height - 1

end while

check that all blocks are saved correctly
return collected blockchain data

To retrieve all Bitcoin blockchain data up to 1 January 2020, a structured backward-iteration approach can be used. First, the block height corresponding to the target date is identified using the blockchain timestamp. This block serves as the starting point for data collection.
From the target block height, blocks are retrieved sequentially in reverse order down to the genesis block. For each block, key information such as the block hash, timestamp, and transaction count is extracted. All transactions within each block are then processed to collect input and output data.
To ensure reliable execution, request rate limits are respected by adding short delays between API calls. After data collection, validation checks are performed to confirm that block heights are continuous and no data is missing. This approach allows efficient historical blockchain reconstruction using public APIs without operating a full Bitcoin node.